# Talkie × WMDP-Bio: local inference smoke test

Uses the original Talkie 1930 **base** model as a 4-bit GGUF on Apple Silicon. The first run downloads about 8.45 GB. This notebook tests output format on eight WMDP-Bio questions; it is not a valid capability estimate. No question text, answer keys, or raw completions are displayed or saved.

## Harmless examples of the four prompt formats
Run the next cell to see the exact text sent to Talkie. The forced A–D diagnostic uses the exam prompt with a one-letter decoding constraint.

In [ ]:
from talkie_poc import format_prompt

example = {
    'question': 'Which month follows January?',
    'choices': ['March', 'June', 'April', 'February'],
}
for style in ('direct', 'exam', 'cloze', 'few_shot'):
    print(f'\n--- {style} ---\n{format_prompt(example, style)}')

In [ ]:
from talkie_poc import download_model, load_model, load_questions, complete, run_pilot, run_constrained_pilot

model_path = download_model()
print(f'Model ready: {model_path.name}')

In [ ]:
model = load_model(model_path)
print('Talkie loaded')

## Benign inference check
A quick check that generation works before evaluating the dataset.

In [ ]:
print(complete(model, 'Question: Which day comes after Monday?\nA. Sunday\nB. Tuesday\nC. Friday\nD. Saturday\nAnswer:'))

## Eight-item WMDP-Bio format pilot
The same items are used for every prompt format. The primary result is the share that produces a standalone A–D letter. Constrained decoding forces a letter, so its parse rate is not a prompt-format result. The initial few-shot run gave 8/8 clean letters, mostly C; accuracy was 2/8 and is only a secondary diagnostic. Talkie's pre-1931 training makes low accuracy plausible, but this pilot cannot establish the cause.

In [ ]:
rows = load_questions(n=8)
print(f'Loaded {len(rows)} questions; content hidden')
results = run_pilot(model, rows) + [run_constrained_pilot(model, rows)]
for result in results:
    print(result)